# Filtered aperture E/B shear maps on HEALPix (DP2, single tract)

Alex Broughton · October 2026. Rubin stack and Butler access required.

## Introduction

Schirmer-filtered aperture E- and B-mode shear maps on HEALPix for one DP2 tract, with shear-peak detection. Shapes are
HSM regauss from the object table. These are weighted aperture statistics, not a Kaiser-Squires reconstruction.

- E-mode: tangential shear g_t weighted by the Schirmer filter Q.
- B-mode: cross shear g× weighted by the same Q; consistent with zero for pure lensing.
- Geometry: every HEALPix cell containing a source in the target tract is evaluated at its center. Galaxies from
  neighboring tracts are included so apertures are complete at the tract edge.
- Shear frame: HSM shapes are rotated from each tract's coadd pixel grid into the local West/North (−RA, +Dec)
  frame at every source with the tract WCS. Positions use the same frame, with exact spherical distances and
  angles from astropy. g× has the standard sign, g× = −Im[g e^(−2iφ)].
- Dust: griz magnitudes are corrected for Milky Way extinction before any color cut, using the Object-table `ebv`
  column (SFD98) and the Rubin R_band coefficients (section 3.0).
- Weights: inverse variance, intrinsic shape noise plus measurement error squared.
- Filter and cells: Rs = 15′, x_c = 0.15, NSIDE = 4096 (0.86′ cells). These maximize S/N for massive clusters
  (M200 ~ 10¹⁵ M⊙) at z ≈ 0.3. For lower-mass clusters (M200 ~ 3×10¹⁴ M⊙) Rs ≈ 6–10′ is closer to optimal, because
  the best aperture scales with r200.
- Peaks: S/N maxima within 0.4 Rs, aperture completeness ≥ 0.8, S/N ≥ 4. They are written to a catalog in output/.

## 0.0 Parameters

`TRACT` is the tract to map. `LOCAL_TABLE` is an optional saved object table for that tract, read when the Rubin
stack is not available. This cell is tagged `parameters` for papermill.

In [ ]:
TRACT = 9813  # DP2 tract to map
LOCAL_TABLE = None  # optional saved object table for TRACT (path relative to _data/ or absolute)

## 1.0 Setup

### 1.1 Imports and configuration

All tunable quantities live in the CONFIG dictionary.

In [ ]:
import gc
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
import astropy.units as u
from astropy.coordinates import SkyCoord, angular_separation, position_angle
from astropy.io import fits
from astropy.table import Table, vstack
from astropy.wcs import WCS
from scipy.spatial import cKDTree
from tqdm.auto import tqdm

# The Rubin stack is optional: without it a saved local object table is read instead.
try:
    from lsst.daf.butler import Butler
    from lsst.sphgeom import Angle as SphAngle, Circle
    from lsst.geom import SpherePoint, degrees
    HAVE_LSST = True
except ImportError:
    HAVE_LSST = False
    print("Rubin stack not found: using CONFIG['LOCAL_TABLE'] instead of the Butler")

%matplotlib inline
plt.rcParams.update({"font.size": 12})


# Repo layout: _data/ holds external inputs, output/ holds generated tables/catalogs, plots/ holds figures.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_DIR = ROOT / "_data"
OUTPUT_DIR = ROOT / "output"
PLOTS_DIR = ROOT / "plots"
for _d in (OUTPUT_DIR, PLOTS_DIR):
    _d.mkdir(parents=True, exist_ok=True)


def save_plot(fig, name):
    fig.savefig(PLOTS_DIR / f"{name}.pdf", bbox_inches="tight")




def tract_area_deg2(skymap, tract):
    """Area of a tract's unique (inner) region in deg^2."""
    region = skymap[tract].inner_sky_region
    if hasattr(region, "getArea"):  # sphgeom.Box, as in rings skymaps
        sr = region.getArea()
    else:  # convex polygon: sum the solid angles of a triangle fan
        v = np.array([[p.x(), p.y(), p.z()] for p in region.getVertices()])
        a = v[0]
        sr = sum(
            2 * np.arctan2(abs(a @ np.cross(b, c)), 1 + a @ b + b @ c + c @ a)
            for b, c in zip(v[1:-1], v[2:])
        )
    return float(sr * (180.0 / np.pi) ** 2)

In [ ]:
CONFIG = {
    "REPO": "/sdf/data/rubin/repo/dp2_prep",  # Butler repository root
    "COLLECTION": "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",  # Input collection
    "SKYMAP": "lsst_cells_v2",  # Skymap name for Butler
    "INSTRUMENT": "LSSTCam",  # Instrument dimension
    "TRACT": int(TRACT),  # Set in the parameters cell
    # Optional saved object table for TRACT, used when the Rubin stack is not importable.
    "LOCAL_TABLE": None if LOCAL_TABLE is None else DATA_DIR / LOCAL_TABLE,
    # --- Filter, cell, and peak settings ---
    "RS_INPUT_PIX": 4500,  # Schirmer aperture radius Rs in native pixels: 4500 x 0.2" = 15'
    "PIX_SCALE_ARCSEC": 0.2,  # Arcsec per pixel (converts RS_INPUT_PIX to sky angle)
    "FILTER_XC": 0.15,  # Schirmer core x_c (NFW-optimized value; the data cannot tell 0.05-0.3 apart)
    "SEARCH_RADIUS_RS_MULT": 1.2,  # Search radius in units of Rs; Q(x) is zero for x > ~1.1
    "NSIDE": int(2**12),  # HEALPix NSIDE (RING), 0.86' cells: worst-case loss 0.7 sigma at the peak
    "PEAK_SN_MIN": 4.0,  # Peak S/N cut, above the random-rotation noise peaks; None disables
    "PEAK_EXCL_RS_MULT": 0.4,  # A peak is the S/N maximum within this many Rs (noise decorrelation length)
    "PEAK_COMPLETENESS_MIN": 0.8,  # Minimum aperture completeness (counts / expected counts)
    "N_RANDOM": 10,  # Random-rotation noise realizations computed alongside the map
    "RANDOM_SEED": 42,
    # --- Shear frame and weights ---
    # Shapes and polar angles both live in the local West/North (-RA, +Dec) frame at each source (see 4.1).
    "SHEAR_FRAME": "local (-RA, +Dec)",
    "SHAPE_NOISE_E": None,  # Intrinsic per-component distortion dispersion; None measures it from the catalog
    # Source selection, following selection_cuts.md.
    "SNR_MIN": 5.0,  # Minimum per-band griz Sersic flux S/N
    "RESO_FACTOR_MIN": 0.125,  # Minimum PSF-resolution factor (star-galaxy separation)
    "COLOR_ABS_MAX": 5.0,  # Maximum |g-r|, |r-i|, |i-z| in AB magnitudes
    "EXTENDEDNESS_MIN": 0.2,  # Minimum griz model extendedness
    "E_SQ_MAX": 4.0,  # Maximum e1^2 + e2^2 for the HSM regauss distortion
    "T_MAX_ARCSEC2": None,  # Optional size cut on i_ixx + i_iyy; None disables it
    "RED_SEQ_SLOPE": 2.0484,  # Red-sequence line: g-i = slope * (r-z) + intercept
    "RED_SEQ_INTERCEPT": -0.10,  # Objects above that line are rejected (dereddened colors when DEREDDEN)
    # Milky Way dust correction, following Rubin tutorial 311.1: m_0 = m - R_band * E(B-V), with E(B-V)
    # from the Object-table ebv column (SFD98). All color cuts use the corrected magnitudes.
    "DEREDDEN": True,
    "EBV_COLUMN": "ebv",
    # R_band = A_band / E(B-V) from rubin_sim DustValues().r_x: CCM89, R_V = 3.1, flat SED, LSST baseline
    # throughputs (rubin_sim 2.6.2, throughputs_2025_08_25).
    "EXTINCTION_R": {"u": 4.757, "g": 3.661, "r": 2.701, "i": 2.054, "z": 1.590, "y": 1.308},
}


USE_BUTLER = HAVE_LSST
if not USE_BUTLER and CONFIG["LOCAL_TABLE"] is None:
    raise RuntimeError("No Rubin stack: set LOCAL_TABLE in the parameters cell to a saved object table for TRACT")

# Filter scale Rs expressed in the angular units needed downstream.
CONFIG["RS_ARCSEC"] = float(CONFIG["RS_INPUT_PIX"]) * float(CONFIG["PIX_SCALE_ARCSEC"])
CONFIG["RS_ARCMIN"] = CONFIG["RS_ARCSEC"] / 60.0
CONFIG["RS_DEG"] = CONFIG["RS_ARCSEC"] / 3600.0

# Aperture radius used for the neighbor search around each HEALPix cell center.
CONFIG["THETA_MAX_ARCMIN"] = float(CONFIG["SEARCH_RADIUS_RS_MULT"]) * CONFIG["RS_ARCMIN"]
CONFIG["NSIDE"] = int(CONFIG["NSIDE"])

# Mean HEALPix pixel spacing, i.e. sqrt(pixel area), as a sanity check against Rs.
_mean_arcsec = float(np.rad2deg((4.0 * np.pi / 12.0) ** 0.5 / CONFIG["NSIDE"]) * 3600.0)
_npix = hp.nside2npix(CONFIG["NSIDE"])

print("TRACT =", CONFIG["TRACT"], "| data source:", "Butler" if USE_BUTLER else CONFIG["LOCAL_TABLE"].name)
print(
    "Schirmer Rs =",
    CONFIG["RS_INPUT_PIX"],
    "pix @",
    CONFIG["PIX_SCALE_ARCSEC"],
    "arcsec/pix →",
    f"{CONFIG['RS_ARCMIN']:.3f} arcmin, x_c = {CONFIG['FILTER_XC']}",
)
print(
    "search radius THETA_MAX_ARCMIN =",
    f"{CONFIG['THETA_MAX_ARCMIN']:.3f}",
    f"({CONFIG['SEARCH_RADIUS_RS_MULT']}×Rs) | NSIDE =",
    CONFIG["NSIDE"],
    f"(npix={_npix}; ~mean spacing {_mean_arcsec:.1f} arcsec)",
)

In [ ]:
# Butler client for the repo/collection defined above (skipped in local-table mode).
butler = None
if USE_BUTLER:
    butler = Butler(
        CONFIG["REPO"],
        collections=[CONFIG["COLLECTION"]],
        skymap=CONFIG["SKYMAP"],
        instrument=CONFIG["INSTRUMENT"],
    )

### 1.2 Table helpers

In [ ]:
def _col(tbl, name):
    """Column as a float64 numpy array."""
    return np.array(tbl[name]).astype(np.float64, copy=False)


def flux_to_abmag(flux_njy):
    """Convert nJy fluxes to AB magnitudes; non-positive or non-finite fluxes become NaN."""
    flux = np.asarray(flux_njy, dtype=np.float64)
    mag = np.full(flux.shape, np.nan, dtype=np.float64)
    good = np.isfinite(flux) & (flux > 0.0)
    mag[good] = (flux[good] * u.nJy).to(u.ABmag).value
    return mag


def griz_mags(tbl):
    """griz AB magnitudes from the Sersic fluxes, dust-corrected when CONFIG["DEREDDEN"] is set.

    m_0 = m - R_band * E(B-V). Objects with a non-finite E(B-V) get NaN magnitudes and fail the color cuts.
    """
    mags = {b: flux_to_abmag(_col(tbl, f"{b}_sersicFlux")) for b in "griz"}
    if not CONFIG["DEREDDEN"]:
        return mags
    ebv_col = CONFIG["EBV_COLUMN"]
    if ebv_col not in tbl.colnames:
        raise KeyError(
            f"CONFIG['DEREDDEN'] is set but the table has no '{ebv_col}' column. Read it from the Butler, "
            "re-save the local table with it, or set CONFIG['DEREDDEN'] = False."
        )
    ebv = _col(tbl, ebv_col)
    return {b: m - CONFIG["EXTINCTION_R"][b] * ebv for b, m in mags.items()}


FLAG_COLUMNS = [
    "i_i_flag",
    "i_iPSF_flag",
    "i_hsmShapeRegauss_flag",
    "i_gaapFlux_flag",
    "sersic_no_data_flag",
    "sersic_unknown_flag",
    "shape_flag",
    "i_pixelFlags_bad",
    "i_pixelFlags_nodata",
    "i_pixelFlags_edge",
    "i_pixelFlags_saturatedCenter",
    "i_pixelFlags_crCenter",
    "i_pixelFlags_interpolatedCenter",
    "i_pixelFlags_suspectCenter",
]

OBJECT_COLUMNS = [
    "objectId",
    "coord_ra",
    "coord_dec",
    "ebv",  # SFD98 E(B-V) at the object position, for the dust correction
    "detect_isDeblendedModelSource",
    "griz_model_extendedness",
    "i_hsmShapeRegauss_e1",
    "i_hsmShapeRegauss_e2",
    "i_hsmShapeRegauss_sigma",
    "i_ixx",
    "i_iyy",
    "i_ixxPSF",
    "i_iyyPSF",
    "i_inputCount",
    "g_sersicFlux",
    "g_sersicFluxErr",
    "r_sersicFlux",
    "r_sersicFluxErr",
    "i_sersicFlux",
    "i_sersicFluxErr",
    "z_sersicFlux",
    "z_sersicFluxErr",
] + FLAG_COLUMNS

SHEAR_COLUMNS = [
    "objectId",
    "coord_ra",
    "coord_dec",
    "i_hsmShapeRegauss_e1",
    "i_hsmShapeRegauss_e2",
    "i_hsmShapeRegauss_sigma",
]


def selection_mask(data):
    """Combined selection mask and the per-group masks that build it."""
    n = len(data)
    flag_columns = [c for c in FLAG_COLUMNS if c in data.colnames]

    flags_ok = np.ones(n, dtype=bool)
    for name in flag_columns:
        flags_ok &= ~np.asarray(data[name], dtype=bool)
    if "i_inputCount" in data.colnames:
        flags_ok &= np.asarray(data["i_inputCount"]) > 0

    primary_ok = np.asarray(data["detect_isDeblendedModelSource"], dtype=bool)

    e_sq = _col(data, "i_hsmShapeRegauss_e1") ** 2 + _col(data, "i_hsmShapeRegauss_e2") ** 2
    shape_ok = e_sq <= CONFIG["E_SQ_MAX"]

    t_obj = _col(data, "i_ixx") + _col(data, "i_iyy")
    with np.errstate(divide="ignore", invalid="ignore"):
        reso = 1.0 - (_col(data, "i_ixxPSF") + _col(data, "i_iyyPSF")) / t_obj
    reso_ok = reso > CONFIG["RESO_FACTOR_MIN"]

    mag = griz_mags(data)  # dust-corrected when CONFIG["DEREDDEN"]
    with np.errstate(divide="ignore", invalid="ignore"):
        snr = {b: _col(data, f"{b}_sersicFlux") / _col(data, f"{b}_sersicFluxErr") for b in "griz"}

    snr_min = CONFIG["SNR_MIN"]
    detect_ok = _col(data, "griz_model_extendedness") > CONFIG["EXTENDEDNESS_MIN"]
    for b in "griz":
        detect_ok &= snr[b] > snr_min

    color_max = CONFIG["COLOR_ABS_MAX"]
    gi = mag["g"] - mag["i"]
    rz = mag["r"] - mag["z"]
    color_ok = np.abs(mag["g"] - mag["r"]) < color_max
    color_ok &= np.abs(mag["r"] - mag["i"]) < color_max
    color_ok &= np.abs(mag["i"] - mag["z"]) < color_max
    color_ok &= np.isfinite(gi) & np.isfinite(rz)
    color_ok &= ~(gi > CONFIG["RED_SEQ_SLOPE"] * rz + CONFIG["RED_SEQ_INTERCEPT"])

    size_ok = np.ones(n, dtype=bool)
    if CONFIG["T_MAX_ARCSEC2"] is not None:
        size_ok = t_obj < CONFIG["T_MAX_ARCSEC2"] / CONFIG["PIX_SCALE_ARCSEC"] ** 2

    groups = {
        "measurement + pixel flags": flags_ok,
        "deblended model source": primary_ok,
        "|e| <= 2": shape_ok,
        f"reso_factor > {CONFIG['RESO_FACTOR_MIN']}": reso_ok,
        f"extendedness + S/N > {snr_min:g}": detect_ok,
        "colors + red sequence": color_ok,
        "size": size_ok,
    }
    mask = np.ones(n, dtype=bool)
    for cut in groups.values():
        mask &= cut
    return mask, groups


def resolve_object_columns(butler, tract):
    """Intersect OBJECT_COLUMNS with the table schema without loading the rows."""
    data_id = {"tract": int(tract), "skymap": CONFIG["SKYMAP"]}
    try:
        names = set(
            butler.get("object", dataId=data_id, storageClass="ArrowSchema").names
        )
    except Exception as exc:
        print("ArrowSchema unavailable, requesting OBJECT_COLUMNS directly:", exc)
        return list(OBJECT_COLUMNS)
    cols = [c for c in OBJECT_COLUMNS if c in names]
    missing = [c for c in OBJECT_COLUMNS if c not in names]
    print("Missing columns:", missing or "none")
    print(f"Reading {len(cols)} of {len(names)} columns")
    return cols


def load_object_table(butler, tract, columns):
    """Object table for one tract, restricted to `columns`."""
    return butler.get(
        "object",
        dataId={"tract": int(tract), "skymap": CONFIG["SKYMAP"]},
        parameters={"columns": columns},
    )


def object_tract_ids(butler):
    """Tract ids that have an object table in the configured collection."""
    data_ids = butler.registry.queryDataIds(
        ["tract"],
        datasets="object",
        collections=CONFIG["COLLECTION"],
        skymap=CONFIG["SKYMAP"],
    )
    return {int(d["tract"]) for d in data_ids}


def buffered_tract_circle(tract_info, theta_max_deg):
    """Bounding circle of the tract outer polygon, expanded by theta_max_deg."""
    bc = tract_info.getOuterSkyPolygon().getBoundingCircle()
    return Circle(
        bc.getCenter(),
        SphAngle(bc.getOpeningAngle().asRadians() + np.deg2rad(theta_max_deg)),
    )


def neighbor_tract_ids(skymap, tract_id, available, theta_max_deg):
    """Available object-table tracts whose outer polygon meets the buffered target tract."""
    buf = buffered_tract_circle(skymap[int(tract_id)], theta_max_deg)
    neighbors = []
    for tid in sorted(available):
        if tid == tract_id:
            continue
        try:
            other = skymap[int(tid)].getOuterSkyPolygon()
            if buf.intersects(other):
                neighbors.append(int(tid))
        except Exception:
            continue
    return neighbors, buf


def in_circle(ra_deg, dec_deg, circle):
    """True for points inside a sphgeom Circle."""
    ra = np.deg2rad(np.asarray(ra_deg, dtype=np.float64))
    dec = np.deg2rad(np.asarray(dec_deg, dtype=np.float64))
    cosd = np.cos(dec)
    c = circle.getCenter()
    try:
        cx, cy, cz = float(c.x()), float(c.y()), float(c.z())
    except TypeError:
        cx, cy, cz = float(c.x), float(c.y), float(c.z)
    cosang = cosd * np.cos(ra) * cx + cosd * np.sin(ra) * cy + np.sin(dec) * cz
    return cosang >= np.cos(circle.getOpeningAngle().asRadians())


def _rad(angle):
    """Plain float radians from an astropy Angle/Quantity or a bare array that is already in radians."""
    return angle.to_value(u.rad) if hasattr(angle, "to_value") else np.asarray(angle, dtype=np.float64)


def tract_wcs(tract):
    """Astropy WCS of a tract's coadd pixel grid, or None if it is not available.

    With the Butler it comes from the skymap and is cached to _data/tract-<id>_wcs.fits, so a later
    local-table run can use it too. In local-table mode only that cached file is used.
    """
    path = DATA_DIR / f"tract-{int(tract)}_wcs.fits"
    if skymap is not None:
        try:
            md = skymap[int(tract)].getWcs().getFitsMetadata()
            hdr = fits.Header()
            for key, val in md.toDict().items():
                hdr[key] = val
            try:
                fits.PrimaryHDU(header=hdr).writeto(path, overwrite=True)
            except OSError as exc:
                print(f"tract {tract}: could not cache the WCS to {path} ({exc})")
            return WCS(hdr)
        except Exception as exc:
            print(f"tract {tract}: could not convert the skymap WCS ({exc})")
    if path.exists():
        return WCS(fits.getheader(path))
    return None


def shapes_to_local_frame(e1, e2, ra, dec, wcs, step_pix=10.0):
    """Rotate pixel-frame ellipticities into the local West/North (-RA, +Dec) frame at each source.

    HSM moments are measured on the tract's coadd pixel grid. Its +x axis points West and +y North only at the
    tract center; elsewhere the axes are rotated by the convergence of meridians, and each neighbor tract has its
    own grid. The direction of each pixel axis at the source is read off the WCS with astropy position angles
    (East of North) and converted to an angle beta measured from West toward North (beta = PA + 90 deg). The
    spin-2 ellipticity then transforms as
        e_local = e_pix * exp(2i beta_x)          if the grid has the same handedness as West/North,
        e_local = conj(e_pix) * exp(2i beta_x)    if it is mirrored.
    Returns (e1_local, e2_local, beta_x in rad); with wcs=None the pixel axes are assumed to point West/North
    (exact only at the tract center) and beta_x is None.
    """
    e1 = np.asarray(e1, dtype=np.float64)
    e2 = np.asarray(e2, dtype=np.float64)
    if wcs is None:
        return e1.copy(), e2.copy(), None
    ra = np.asarray(ra, dtype=np.float64)
    dec = np.asarray(dec, dtype=np.float64)
    x, y = wcs.all_world2pix(ra, dec, 0)
    ra_x, dec_x = wcs.all_pix2world(x + step_pix, y, 0)  # one step along pixel +x
    ra_y, dec_y = wcs.all_pix2world(x, y + step_pix, 0)  # one step along pixel +y
    r = np.deg2rad
    beta_x = _rad(position_angle(r(ra), r(dec), r(ra_x), r(dec_x))) + 0.5 * np.pi
    beta_y = _rad(position_angle(r(ra), r(dec), r(ra_y), r(dec_y))) + 0.5 * np.pi
    mirrored = np.sin(beta_y - beta_x) < 0.0
    e = e1 + 1j * e2
    e = np.where(mirrored, np.conj(e), e) * np.exp(2j * beta_x)
    return e.real, e.imag, beta_x


def shear_subset(tbl, tract):
    """Keep only the columns needed after selection, plus e1_local/e2_local: the HSM shapes rotated from the
    pixel grid of `tract` into the local West/North frame at each source (see shapes_to_local_frame)."""
    e1l, e2l, beta = shapes_to_local_frame(
        _col(tbl, "i_hsmShapeRegauss_e1"), _col(tbl, "i_hsmShapeRegauss_e2"),
        _col(tbl, "coord_ra"), _col(tbl, "coord_dec"), tract_wcs(tract),
    )
    if beta is None:
        print(f"tract {tract}: no WCS available, assuming the pixel axes point West/North (exact only at the tract center)")
    elif len(tbl):
        rot = np.rad2deg(np.angle(np.exp(1j * beta)))  # pixel +x relative to West, wrapped to (-180, 180]
        print(f"tract {tract}: pixel -> local West/North rotation: median {np.median(rot):+.3f} deg, "
              f"range [{rot.min():+.3f}, {rot.max():+.3f}] deg")
    out = tbl[[c for c in SHEAR_COLUMNS if c in tbl.colnames]]
    out["e1_local"] = e1l
    out["e2_local"] = e2l
    return out


def dedupe_objectid(tbl, n_target):
    """Drop duplicate objectId rows, keeping the target-tract copy when both exist."""
    if "objectId" not in tbl.colnames:
        from_target = np.arange(len(tbl)) < n_target
        return tbl, from_target
    ids = np.asarray(tbl["objectId"])
    _, first = np.unique(ids, return_index=True)
    first.sort()
    return tbl[first], first < n_target

### 1.3 Filter, aperture, and peak functions

In [ ]:
def wrap_dra(ra, ra0):
    """RA offset ra - ra0 in degrees, wrapped to [-180, 180) so offsets stay correct across RA = 0."""
    return (np.asarray(ra, dtype=np.float64) - ra0 + 180.0) % 360.0 - 180.0


def radec_to_xyz(ra, dec):
    """Unit vectors (n, 3) for RA/Dec in degrees, for seam-free KD-tree searches."""
    ra = np.deg2rad(np.atleast_1d(np.asarray(ra, dtype=np.float64)))
    dec = np.deg2rad(np.atleast_1d(np.asarray(dec, dtype=np.float64)))
    return np.column_stack([np.cos(dec) * np.cos(ra), np.cos(dec) * np.sin(ra), np.sin(dec)])


def chord_radius(theta_deg, pad=1.1):
    """Chord length on the unit sphere for an angular radius, padded so the KD-tree query is a
    superset of the exact angular cut redone inside the aperture function."""
    return 2.0 * np.sin(np.deg2rad(pad * float(theta_deg)) / 2.0)


def circular_mean_ra(ra):
    """Mean RA in degrees on [0, 360), correct for sets that straddle RA = 0."""
    ra_rad = np.deg2rad(np.asarray(ra, dtype=np.float64))
    return float(np.rad2deg(np.arctan2(np.mean(np.sin(ra_rad)), np.mean(np.cos(ra_rad)))) % 360.0)


def ra_for_plot(ra, ra_ref):
    """RA made continuous around ra_ref, so a tract crossing RA = 0 plots as one patch (values may be < 0 or > 360)."""
    return float(ra_ref) + wrap_dra(ra, ra_ref)


def ra_axis(ax):
    """RA x-axis: label, tick labels wrapped to [0, 360), and RA increasing to the left."""
    from matplotlib.ticker import FuncFormatter

    ax.set_xlabel("RA (deg)")
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x % 360.0:g}"))
    ax.invert_xaxis()


def schirmer_weight(r, rs, xc=None):
    """Schirmer filter Q(r/Rs); r and rs in the same angular units (degrees here). xc defaults to CONFIG."""
    r = np.asarray(r, dtype=np.float64)
    rs = float(rs)
    x = r / rs

    # Schirmer (2007) coefficients: a smooth band-pass in x with a tanh core.
    a, b, c, d = 6.0, 150.0, 47.0, 50.0
    xc = CONFIG["FILTER_XC"] if xc is None else float(xc)

    # Outer envelope: rises near x ~ a/b and falls near x ~ c/d.
    q = 1.0 / (1.0 + np.exp(a - b * x) + np.exp(d * x - c))

    # Core term tanh(x/xc)/(x/xc), which tends to 1 as x -> 0.
    ratio = x / xc
    with np.errstate(divide="ignore", invalid="ignore"):
        inner = np.tanh(ratio) / ratio
    inner = np.where(np.abs(ratio) < 1e-14, 1.0, inner)

    q = q * inner
    q = np.where(np.isfinite(q), q, 0.0)  # Guard against 0/0 at the center
    return q


def ang_sep_deg(ra, dec, center):
    """Exact great-circle distance in degrees from center = (ra0, dec0), via astropy (no flat-sky approximation)."""
    ra = np.deg2rad(np.asarray(ra, dtype=np.float64))
    dec = np.deg2rad(np.asarray(dec, dtype=np.float64))
    ra0, dec0 = np.deg2rad(float(center[0])), np.deg2rad(float(center[1]))
    return np.rad2deg(_rad(angular_separation(ra0, dec0, ra, dec)))


def sep_and_phi(ra, dec, center):
    """Exact separation theta (deg) and polar angle phi (rad) of each source about center = (ra0, dec0).

    Both come from astropy's spherical trigonometry, so there is no single-cos(dec0) approximation and no
    RA = 0 seam. phi is the direction pointing away from the center, measured at the source in its local
    West/North (-RA, +Dec) frame, the same frame as the shapes (see shapes_to_local_frame). astropy position
    angles are East of North, and a direction with position angle PA has West/North polar angle PA + 90 deg.
    """
    ra = np.deg2rad(np.asarray(ra, dtype=np.float64))
    dec = np.deg2rad(np.asarray(dec, dtype=np.float64))
    ra0, dec0 = np.deg2rad(float(center[0])), np.deg2rad(float(center[1]))
    theta = np.rad2deg(_rad(angular_separation(ra0, dec0, ra, dec)))
    pa_out = _rad(position_angle(ra, dec, ra0, dec0)) + np.pi  # (source -> center) + 180 deg = outward
    return theta, pa_out + 0.5 * np.pi


def gamma_tx_sky(ra, dec, g1, g2, center, phi=None):
    """Tangential g_t and cross g_x shear about center = (ra0, dec0), in the standard convention

        g_t = -Re[g exp(-2i phi)] = -(g1 cos 2phi + g2 sin 2phi)
        g_x = -Im[g exp(-2i phi)] = -(-g1 sin 2phi + g2 cos 2phi)

    g1, g2 must be in the local West/North frame of shapes_to_local_frame, the frame phi is measured in.
    g_t does not depend on the handedness of the frame, but g_x changes sign under a mirror flip, so the
    frame is part of the convention. Pass phi (from sep_and_phi) to skip recomputing it.
    """
    g1 = np.asarray(g1, dtype=np.float64)
    g2 = np.asarray(g2, dtype=np.float64)
    if phi is None:
        _, phi = sep_and_phi(ra, dec, center)
    c2 = np.cos(2.0 * phi)
    s2 = np.sin(2.0 * phi)
    gt = -(g1 * c2 + g2 * s2)
    gx = -(-g1 * s2 + g2 * c2)
    return gt, gx


Schirmer (2007) filter Q(r/Rs) used to weight gt and g×. Beyond x ≈ c/d = 0.94 the filter is cut off sharply, so Rs is in practice the
aperture radius, and a search disc of 1.2 Rs captures all the weight. Inside the core (x < x_c = 0.15) Q is flat. Outside it Q ∝ 1/x,
the matched filter for an isothermal (SIS) shear profile g_t ∝ 1/θ. The steep term exp(6 − 150x) removes the innermost ~0.04 Rs,
where cluster members, blending, and the breakdown of the weak-shear approximation are worst.

In [ ]:
# Filter profile vs r/Rs, with Rs and the aperture search radius marked.
rs_arcmin = float(CONFIG["RS_ARCMIN"])
x_max = float(CONFIG["SEARCH_RADIUS_RS_MULT"]) * 1.15
x = np.linspace(1e-4, x_max, 800)
q = schirmer_weight(x * CONFIG["RS_DEG"], CONFIG["RS_DEG"])

fig_q, ax_q = plt.subplots(figsize=(7.0, 4.0))
ax_q.plot(x, q, "k-", lw=1.8, label=r"Q(r/Rs)")
ax_q.axvline(1.0, color="C0", ls="--", lw=1.0, label=fr"Rs = {rs_arcmin:.2f}'")
ax_q.axvline(
    CONFIG["SEARCH_RADIUS_RS_MULT"],
    color="C3",
    ls=":",
    lw=1.2,
    label=fr"search = {CONFIG['SEARCH_RADIUS_RS_MULT']}Rs",
)
ax_q.set_xlabel(r"r/Rs")
ax_q.set_ylabel(r"Q(r/Rs)")
ax_q.set_xlim(0, x_max)
ax_q.set_ylim(bottom=0)
ax_q.legend(framealpha=1)
ax_q.set_title("Schirmer aperture-mass filter")

ax_r = ax_q.twiny()
ax_r.set_xlim(0, x_max * rs_arcmin)
ax_r.set_xlabel(r"r (arcmin)")
fig_q.tight_layout()
save_plot(fig_q, "schirmer_filter")

In [ ]:
def aperture_mass_and_sn_at_center(
    ra_c,
    dec_c,
    ra_all,
    dec_all,
    g1_all,
    g2_all,
    *,
    theta_max_arcmin,
    rs_deg,
    w_all=None,
    xc=None,
    g1_rand=None,
    g2_rand=None,
):
    """Filtered aperture E- and B-mode statistics and S/N at one center; optional w_all weights galaxies.

    Returns (ap_e, sn_e, ap_b, sn_b, sn_rand). sn_rand holds the E-mode S/N for each random-rotation
    realization, from g1_rand/g2_rand (shape n_gal x K), or is None when those are not given. A
    realization rotates every galaxy by a fixed random angle (the same angle in every aperture), which
    removes the lensing signal and keeps the shape noise and source positions.
    """
    nan5 = (np.nan, np.nan, np.nan, np.nan, None)
    theta_max_deg = float(theta_max_arcmin) / 60.0
    # Exact great-circle distance and polar angle of every source (astropy): Q(r) uses the true angular r,
    # and there is no RA = 0 seam or single-cos(dec0) approximation.
    dtheta, phi_all = sep_and_phi(ra_all, dec_all, (ra_c, dec_c))

    # Restrict to sources inside the aperture; the KD-tree pre-selection is a superset of this.
    m = dtheta <= theta_max_deg
    if not np.any(m):
        return nan5

    ra_s = ra_all[m]
    dec_s = dec_all[m]
    g1_s = g1_all[m]
    g2_s = g2_all[m]
    d_theta = dtheta[m]
    phi_s = phi_all[m]
    n_g = int(ra_s.size)

    # Uniform weights unless per-object inverse variances are supplied.
    w_s = np.ones(n_g, dtype=np.float64) if w_all is None else np.asarray(w_all, dtype=np.float64)[m]

    gt, gx = gamma_tx_sky(ra_s, dec_s, g1_s, g2_s, center=(ra_c, dec_c), phi=phi_s)

    Qi = schirmer_weight(d_theta, rs_deg, xc)
    w_sum = float(np.sum(w_s))
    if w_sum <= 0.0 or not np.isfinite(w_sum):
        return nan5

    # Weighted aperture statistics: E-mode from g_t, B-mode from g_x.
    ap_e = np.sum(Qi * gt * w_s) / w_sum
    ap_b = np.sum(Qi * gx * w_s) / w_sum

    # Shape-noise error estimate, sum_i Q_i^2 |e_i|^2 w_i^2, giving S/N = sqrt(2) * signal / noise.
    denom = np.sqrt(np.sum((Qi**2) * (g1_s**2 + g2_s**2) * (w_s**2)))
    if denom <= 0.0 or not np.isfinite(denom):
        return ap_e, np.nan, ap_b, np.nan, None
    sn_e = np.sqrt(2.0) * (w_sum * ap_e) / denom
    sn_b = np.sqrt(2.0) * (w_sum * ap_b) / denom

    sn_rand = None
    if g1_rand is not None:
        # |g| is unchanged by rotation, so the noise denominator is shared by all realizations.
        c2, s2 = np.cos(2.0 * phi_s)[:, None], np.sin(2.0 * phi_s)[:, None]
        gt_r = -(g1_rand[m] * c2 + g2_rand[m] * s2)
        sn_rand = np.sqrt(2.0) * ((Qi * w_s) @ gt_r) / denom
    return ap_e, sn_e, ap_b, sn_b, sn_rand

In [ ]:
def find_peaks_shear_healpix(
    ipix_eval,
    sn_map,
    completeness,
    nside,
    excl_radius_arcmin,
    sn_min=None,
    completeness_min=0.8,
):
    """Peaks of an S/N map on HEALPix cells.

    A cell is a peak if (1) its S/N is the highest among all evaluated cells within excl_radius_arcmin,
    (2) its aperture completeness is at least completeness_min, and (3) its S/N is at least sn_min.
    Using S/N rather than M_ap means a noisy shallow region does not produce large-|M_ap| peaks. An
    exclusion radius of about the noise decorrelation length (~0.4 Rs) stops one noise bump from
    being counted several times. Returns the HEALPix indices of the peaks, highest S/N first.
    """
    ipix_eval = np.asarray(ipix_eval, dtype=np.int64)
    sn_map = np.asarray(sn_map, dtype=np.float64)
    completeness = np.asarray(completeness, dtype=np.float64)

    # ipix_eval is sorted (np.unique), so HEALPix index -> array position is a searchsorted lookup.
    srt = np.argsort(ipix_eval)
    ip_sorted = ipix_eval[srt]

    cand = np.isfinite(sn_map) & (completeness >= completeness_min)
    if sn_min is not None:
        cand &= sn_map >= sn_min
    order = np.flatnonzero(cand)
    order = order[np.argsort(-sn_map[order])]

    excl_rad = np.deg2rad(float(excl_radius_arcmin) / 60.0)
    peaks = []
    for k in order:
        vec = hp.pix2vec(nside, int(ipix_eval[k]), nest=False)
        disc = hp.query_disc(nside, vec, excl_rad, inclusive=False, nest=False)
        pos = np.clip(np.searchsorted(ip_sorted, disc), 0, ip_sorted.size - 1)
        j = srt[pos[ip_sorted[pos] == disc]]
        vals = sn_map[j]
        if np.all(~np.isfinite(vals) | (vals <= sn_map[k]) | (j == k)):
            peaks.append(int(ipix_eval[k]))
    return np.asarray(peaks, dtype=np.int64)

## 2.0 Load data

In [ ]:
theta_max_deg = CONFIG["THETA_MAX_ARCMIN"] / 60.0
NEIGHBOR_TRACTS, TRACT_BUFFER_CIRCLE = [], None

if USE_BUTLER:
    object_columns = resolve_object_columns(butler, CONFIG["TRACT"])
    available_tracts = object_tract_ids(butler)
    skymap = butler.get("skyMap", dataId={"skymap": CONFIG["SKYMAP"]})


    try:
        NEIGHBOR_TRACTS, TRACT_BUFFER_CIRCLE = neighbor_tract_ids(
            skymap, CONFIG["TRACT"], available_tracts, theta_max_deg
        )
    except Exception as exc:
        print("Neighbor lookup failed, mapping the target tract only:", exc)

    data = load_object_table(butler, CONFIG["TRACT"], object_columns)
else:
    skymap = None
    data = Table.read(CONFIG["LOCAL_TABLE"])
    object_columns = [c for c in OBJECT_COLUMNS if c in data.colnames]

print(f"Target tract {CONFIG['TRACT']}: {len(data):,} rows")
print("Neighbor tracts:", NEIGHBOR_TRACTS or "none")

## 3.0 Source selection

The cuts follow selection_cuts.md, which is written for the metadetect catalog
(object_shear_all). Here we use the object table and HSM regauss shapes, so each cut
is applied through its object-table counterpart:

| selection_cuts.md (metadetect) | Used here (object table) |
| --- | --- |
| all *_flags false | i-band measurement, PSF, shape, GAaP, and Sersic flags |
| bmask_flags, image_flags, mfrac < 0.1 | i-band pixelFlags_* (bad, nodata, edge, saturatedCenter, crCenter, interpolatedCenter, suspectCenter) |
| is_primary | detect_isDeblendedModelSource |
| SNR > 10 | griz Sersic flux S/N > SNR_MIN (set in CONFIG) |
| reso_factor > 0.5 | 1 - T_PSF/T_obj from the i-band second moments, > RESO_FACTOR_MIN (set in CONFIG) |
| \|g-r\|, \|r-i\|, \|i-z\| < 5 | same, on AB magnitudes from the Sersic fluxes |
| gauss_T < 20 | optional cut on i_ixx + i_iyy, disabled by default |

There is no per-object masked-pixel fraction in the object table, so mfrac is covered only
indirectly by the pixel flags. Two cuts specific to this analysis are added on top: a bound on
the HSM distortion, and a red-sequence veto.

All color cuts use magnitudes corrected for Milky Way dust, $m_0 = m - R_{\rm band}\,E(B-V)$, following Rubin
tutorial 311.1. E(B−V) is the Object-table `ebv` column (SFD98 at each object's position) and R_band comes from
rubin_sim `DustValues().r_x` (CCM89, R_V = 3.1, flat SED): R_g, R_r, R_i, R_z = 3.661, 2.701, 2.054, 1.590.
Without the correction, reddening moves red-sequence galaxies below the veto line by about
(R_g − R_i) − 2.0484 (R_r − R_z) ≈ −0.67 E(B−V) mag in g−i, so they leak into the source sample in dusty tracts.
Set `CONFIG["DEREDDEN"] = False` to turn the correction off.

### 3.1 Per-object quantities

In [ ]:
# Sersic fluxes and their errors, in nJy.
g_flux, g_err = _col(data, "g_sersicFlux"), _col(data, "g_sersicFluxErr")
r_flux, r_err = _col(data, "r_sersicFlux"), _col(data, "r_sersicFluxErr")
i_flux, i_err = _col(data, "i_sersicFlux"), _col(data, "i_sersicFluxErr")
z_flux, z_err = _col(data, "z_sersicFlux"), _col(data, "z_sersicFluxErr")

# AB magnitudes, dust-corrected when CONFIG["DEREDDEN"]; non-positive fluxes become NaN and are dropped by the
# color cuts below.
_mags = griz_mags(data)
g_mag, r_mag, i_mag, z_mag = (_mags[b] for b in "griz")
if CONFIG["DEREDDEN"]:
    _ebv = _col(data, CONFIG["EBV_COLUMN"])
    print(
        f"E(B-V) in tract {CONFIG['TRACT']}: median {np.nanmedian(_ebv):.3f}, "
        f"range {np.nanmin(_ebv):.3f} to {np.nanmax(_ebv):.3f}"
    )
    print("A_band at the median:", {b: round(CONFIG['EXTINCTION_R'][b] * float(np.nanmedian(_ebv)), 3) for b in 'griz'})

# Adjacent-band colors for the sanity cuts, plus (g-i, r-z) for the red-sequence line.
gr = g_mag - r_mag
ri = r_mag - i_mag
iz = i_mag - z_mag
gi = g_mag - i_mag
rz = r_mag - z_mag

# Per-band detection significance.
g_snr = g_flux / g_err
r_snr = r_flux / r_err
i_snr = i_flux / i_err
z_snr = z_flux / z_err

# Resolution factor: 1 - (PSF second moments) / (object second moments).
# Values near 0 are PSF-like, so stars sit near the bottom of the diagnostic below.
reso_factor = 1 - (_col(data, "i_ixxPSF") + _col(data, "i_iyyPSF")) / (
    _col(data, "i_ixx") + _col(data, "i_iyy")
)

### 3.2 Diagnostics

Color-color and star-galaxy separation planes, with the cut thresholds drawn on top.

In [ ]:
# Color-color plane before any cuts, with the red-sequence veto line from CONFIG.
_cc_tag = ", dereddened" if CONFIG["DEREDDEN"] else ""

def cc_seq_parallel(rz):
    return CONFIG["RED_SEQ_SLOPE"] * rz + CONFIG["RED_SEQ_INTERCEPT"]

plt.hexbin(rz, gi, mincnt=1, extent=(-1,3,-1,3))
s = np.linspace(-5,5, 1000)
plt.plot(s, cc_seq_parallel(s), "r-")
plt.ylabel(f"g - i (AB mag{_cc_tag})")
plt.xlabel(f"r - z (AB mag{_cc_tag})")
plt.colorbar()
plt.xlim(-1,3)
plt.ylim(-1,3)

In [ ]:
# Star-galaxy separation plane; the dashed lines mark the cuts applied in 3.3.
plt.plot(i_snr, reso_factor, ',')
plt.axhline(CONFIG["RESO_FACTOR_MIN"], ls='--', color='k', alpha=0.5)
plt.axvline(CONFIG["SNR_MIN"], ls='--', color='k', alpha=0.5)
plt.xlabel("i-band Sersic flux S/N")
plt.ylabel("resolution factor")
plt.ylim(-0.5, 1.0)
plt.xlim(1, 1000)
plt.semilogx()
save_plot(plt.gcf(), f"{CONFIG['TRACT']}_star_galaxy")

### 3.3 Apply the cuts

The red-sequence veto removes galaxies above a line of slope 2.0484 (intercept −0.10) in the dereddened (r−z, g−i) plane. These are likely
cluster members rather than lensed background sources, so they would dilute the signal.

In [ ]:
mask, groups = selection_mask(data)
shear_catalog = data[mask]
from_target = np.ones(len(shear_catalog), dtype=bool)

for _name, _cut in groups.items():
    print(f"{_name:<28} {int(np.sum(_cut)):>10,}")
print(f"{'all cuts':<28} {len(shear_catalog):>10,} of {len(data):,}")

In [ ]:
# Source density over the tract's unique (inner) area. Without the skymap, the area is the
# footprint of occupied NSIDE=1024 (3.4') cells, which slightly overestimates the area at the edges.
if skymap is not None:
    tract_area = tract_area_deg2(skymap, CONFIG["TRACT"])
else:
    _ip = hp.ang2pix(1024, _col(shear_catalog, "coord_ra"), _col(shear_catalog, "coord_dec"), lonlat=True)
    tract_area = np.unique(_ip).size * hp.nside2pixarea(1024, degrees=True)
print(f"tract area: {tract_area:.3f} deg^2")
N_DENSITY = len(shear_catalog) / (tract_area * 3600.0)  # gal / arcmin^2, used for aperture completeness
print(round(N_DENSITY, 3), "gal / arcmin^2")

In [ ]:
# Same color-color plane after the cuts: the red sequence should be removed.
_mags_sel = griz_mags(shear_catalog)  # same (dust-corrected) magnitudes as the cuts
plt.hexbin(
    _mags_sel["r"] - _mags_sel["z"],
    _mags_sel["g"] - _mags_sel["i"], mincnt=1, extent=(-1,3,-1,3))
s = np.linspace(-5,5, 1000)
plt.plot(s, cc_seq_parallel(s), "r-")
plt.ylabel(f"g - i (AB mag{_cc_tag})")
plt.xlabel(f"r - z (AB mag{_cc_tag})")
plt.colorbar()
plt.xlim(-1,3)
plt.ylim(-1,3)

Number of selected galaxies per HEALPix cell after the cuts.

In [ ]:
# Occupied HEALPix cells after the selection cuts (target tract only).
nside_gal = int(CONFIG["NSIDE"])
ra_sel = _col(shear_catalog, "coord_ra")
dec_sel = _col(shear_catalog, "coord_dec")
ipix_sel = hp.ang2pix(nside_gal, ra_sel, dec_sel, nest=False, lonlat=True)
ipix_occ, n_gal_cell = np.unique(ipix_sel, return_counts=True)
ra_cell, dec_cell = hp.pix2ang(nside_gal, ipix_occ, nest=False, lonlat=True)
_ra_ref = circular_mean_ra(ra_cell)  # Plot RA continuously around the tract, even across RA = 0

print(
    f"Occupied cells: {ipix_occ.size:,}  |  "
    f"galaxies/cell: min={int(n_gal_cell.min())}  "
    f"median={np.median(n_gal_cell):.0f}  "
    f"mean={n_gal_cell.mean():.1f}  "
    f"max={int(n_gal_cell.max())}"
)

fig_ngal, axes_ngal = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
pt_ngal = max(1.0, min(8.0, 500_000.0 / max(int(ipix_occ.size), 1)))
sc_ngal = axes_ngal[0].scatter(
    ra_for_plot(ra_cell, _ra_ref),
    dec_cell,
    c=n_gal_cell,
    s=pt_ngal,
    cmap="magma",
    linewidths=0,
    alpha=0.85,
)
ra_axis(axes_ngal[0])
axes_ngal[0].set_ylabel("Dec (deg)")
axes_ngal[0].set_title(f"Galaxies per HEALPix cell (NSIDE={nside_gal})")
fig_ngal.colorbar(sc_ngal, ax=axes_ngal[0], fraction=0.046, pad=0.02, label="N galaxies")

n_max = int(n_gal_cell.max())
bins_ngal = np.arange(0.5, n_max + 1.5, 1) if n_max < 80 else 40
axes_ngal[1].hist(n_gal_cell, bins=bins_ngal, histtype="stepfilled", color="0.35")
axes_ngal[1].set_xlabel("Galaxies per cell")
axes_ngal[1].set_ylabel("Number of cells")
axes_ngal[1].set_title("Occupancy after selection cuts")
save_plot(fig_ngal, f"{CONFIG['TRACT']}_ngal_cells")

### 3.4 Neighbor-tract buffer

Selected galaxies from neighboring tracts within the search radius of the target tract.

In [ ]:
n_target = len(shear_catalog)
pieces = [shear_subset(shear_catalog, CONFIG["TRACT"])]
del shear_catalog, data
gc.collect()

for tid in NEIGHBOR_TRACTS:  # empty in local-table mode
    try:
        tbl = load_object_table(butler, tid, object_columns)
    except Exception as exc:
        print(f"tract {tid}: skipped ({exc})")
        continue
    nb_mask, _ = selection_mask(tbl)
    sel = shear_subset(tbl[nb_mask], tid)
    del tbl
    if len(sel) == 0:
        gc.collect()
        continue
    keep = in_circle(sel["coord_ra"], sel["coord_dec"], TRACT_BUFFER_CIRCLE)
    n_keep = int(np.sum(keep))
    if n_keep:
        pieces.append(sel[keep])
    del sel
    gc.collect()
    print(f"tract {tid}: {n_keep:,} galaxies in buffer")

shear_catalog = vstack(pieces, join_type="inner") if len(pieces) > 1 else pieces[0]
del pieces
gc.collect()

shear_catalog, from_target = dedupe_objectid(shear_catalog, n_target)
print(
    f"Shear catalog: {int(from_target.sum()):,} in tract, "
    f"{int((~from_target).sum()):,} from neighbors "
    f"({len(shear_catalog):,} unique)"
)

## 4.0 Map on HEALPix

### 4.1 Shear calibration, frame, and weights

Regauss distortions are converted to shear with the responsivity R and multiplicative bias
m calibrated on Abell 360: $g = e / [2R(1+m)]$.

**Shear frame.** `i_hsmShapeRegauss_e1/e2` are measured from pixel moments on each tract's coadd, so they are
defined in that tract's pixel frame. Its +x axis points toward −RA (West) and +y toward +Dec (North), but exactly so
only at the tract center. Elsewhere the axes are rotated by the convergence of the meridians (by about
0.85°·|tan δ| at the tract edge, ≈ 0.8° at δ = −44°), and each neighbor tract has its own pixel grid. A rotation by α
leaks a fraction ≈ 2α of $g_t$ into $g_\times$. In 3.4 every shape is therefore rotated into the local West/North
frame at its own position (`shapes_to_local_frame`). The pixel-axis directions are read off the tract WCS with astropy
position angles, and the spin-2 ellipticity is rotated by twice that angle (and conjugated if the grid were mirrored).
The polar angle φ of each source about an aperture center is measured in the same local West/North frame
(`sep_and_phi`: astropy `position_angle`, which is East of North, converted with φ = PA + 90°). Distances are exact
great-circle separations (astropy `angular_separation`), so there is no single-cos δ₀ flat-sky approximation and no
RA = 0 seam. Shapes and positions must share a frame. With a mirror flip between them, a purely tangential field
($g_1 = -g_t\cos2\phi$, $g_2 = -g_t\sin2\phi$) is measured as $g_t\cos4\phi$ in the E-mode and $g_t\sin4\phi$ in the
B-mode, and the lensing signal averages to zero in both maps.

If the tract WCS is not available (local-table mode without a cached `_data/tract-<id>_wcs.fits`, which a Butler run
writes), the pixel axes are assumed to point exactly West/North, and the run prints a warning.

**Sign convention.** $g_t = -{\rm Re}[g\,e^{-2i\phi}] = -(g_1\cos2\phi + g_2\sin2\phi)$ and
$g_\times = -{\rm Im}[g\,e^{-2i\phi}] = g_1\sin2\phi - g_2\cos2\phi$. $g_t$ does not depend on the handedness of the
frame, but $g_\times$ changes sign under a mirror flip. The West/North frame has the handedness of the coadd pixel
grid, so $g_\times$ here matches pixel-coordinate pipelines (A360, LoVoCCS). To compare with a catalog whose shapes are
defined with x toward +RA (East), flip the sign of its $e_2$ (or of its $g_\times$) first.

**Weights.** $w = 1/(\sigma_{e,\rm int}^2 + \sigma_e^2)$, where $\sigma_e$ is `i_hsmShapeRegauss_sigma`, the per-component
distortion error. The intrinsic dispersion is estimated from the catalog as
$\sigma_{e,\rm int}^2 = \langle e_1^2+e_2^2\rangle/2 - {\rm median}(\sigma_e^2)$.

In [ ]:
# Quick check that the mean ellipticity is small; a large value signals additive bias.
np.mean(shear_catalog['i_hsmShapeRegauss_e1'])

In [ ]:
# Responsivity and multiplicative bias from the A360 calibration.
R, m_mean = 0.8511839657185559, -0.13584282915829576

ra_deg = _col(shear_catalog, "coord_ra")
dec_deg = _col(shear_catalog, "coord_dec")

# Distortion -> shear: g = e / (2R) / (1 + m), from the shapes already rotated into each source's
# local West/North frame (shear_subset in 3.4), the same frame the polar angles are measured in.
e1_0 = _col(shear_catalog, "e1_local")
e2_0 = _col(shear_catalog, "e2_local")
g1 = (e1_0 / (2.0 * R)) / (1.0 + m_mean)
g2 = (e2_0 / (2.0 * R)) / (1.0 + m_mean)

# Weights: 1 / (intrinsic distortion variance + measurement variance), both per component.
sig_e = _col(shear_catalog, "i_hsmShapeRegauss_sigma")
if CONFIG["SHAPE_NOISE_E"] is None:
    _s2 = np.nanmean(e1_0**2 + e2_0**2) / 2.0 - np.nanmedian(sig_e**2)
    CONFIG["SHAPE_NOISE_E"] = float(np.sqrt(max(_s2, 0.05)))
_var_sum = CONFIG["SHAPE_NOISE_E"] ** 2 + sig_e**2
with np.errstate(divide="ignore", invalid="ignore"):
    w_gal = np.where(np.isfinite(_var_sum) & (_var_sum > 0.0), 1.0 / _var_sum, 0.0)

# Effective source density and per-component shear dispersion.
N_EFF = N_DENSITY * np.sum(w_gal[from_target]) ** 2 / np.sum(w_gal[from_target] ** 2) / from_target.sum()
SIGMA_G = float(np.sqrt(np.sum(w_gal**2 * (g1**2 + g2**2) / 2) / np.sum(w_gal**2)))

print("N galaxies:", ra_deg.size)
print("Galaxies with positive finite weight:", int(np.sum(w_gal > 0)))
print(f"sigma_e,int (distortion, per component) = {CONFIG['SHAPE_NOISE_E']:.3f}")
print(f"n_eff = {N_EFF:.1f} / arcmin^2,  sigma_g (shear, per component, weighted) = {SIGMA_G:.3f}")

In [ ]:
# Effective source density from the weights above (target-tract sources only).
_t = from_target & (w_gal > 0)
_w = w_gal[_t]
_area = tract_area * 3600.0  # arcmin^2
sig_int = CONFIG["SHAPE_NOISE_E"] / (2.0 * R * (1.0 + m_mean))  # intrinsic shape noise, shear per component

# Heymans+12: n_eff = (sum w)^2 / (A sum w^2), paired with the weighted shear dispersion.
n_eff_H = np.sum(_w) ** 2 / np.sum(_w**2) / _area
sig_H = np.sqrt(np.sum(_w**2 * (g1[_t] ** 2 + g2[_t] ** 2) / 2) / np.sum(_w**2))

# Chang+13: n_eff = sum_i sigma_int^2 / (sigma_int^2 + sigma_i^2) / A, paired with sigma_int.
n_eff_C = CONFIG["SHAPE_NOISE_E"] ** 2 * np.sum(_w) / _area

print(f"raw n = {_t.sum() / _area:.2f} / arcmin^2")
print(f"Heymans+12: n_eff = {n_eff_H:.2f} / arcmin^2, sigma = {sig_H:.3f}, sigma^2/n_eff = {sig_H**2 / n_eff_H:.2e} arcmin^2")
print(f"Chang+13:   n_eff = {n_eff_C:.2f} / arcmin^2, sigma = {sig_int:.3f}, sigma^2/n_eff = {sig_int**2 / n_eff_C:.2e} arcmin^2")
print(f"equivalent n at sigma = 0.25 (Frieman 2026): {0.25**2 * n_eff_H / sig_H**2:.2f} / arcmin^2")

### 4.3 Aperture statistic on every cell

`N_RANDOM` noise realizations are computed alongside the map. In each, every galaxy is rotated by a random angle that
stays fixed across apertures. These maps keep the real positions, weights, masks, and depth variations but contain no
lensing signal.

Aperture completeness is the number of sources within the search disc divided by `N_DENSITY · π θ_max²`. It is low
near tract edges when neighbors are missing, and around large masks.

In [ ]:
def run_aperture_map(nside, rs_deg, theta_max_arcmin, xc=None, n_random=0, eval_mask=None, seed=0, progress=True):
    """Evaluate the aperture statistic on every occupied HEALPix cell of the target tract.

    Returns a dict with ipix, ap_e, sn_e, ap_b, sn_b, counts, completeness and sn_rand (n_cell x n_random).
    eval_mask optionally restricts the evaluated cells (boolean over the occupied cells).
    """
    theta_max_deg = theta_max_arcmin / 60.0
    ipix_all = hp.ang2pix(nside, ra_deg, dec_deg, nest=False, lonlat=True)
    ipix_ev = np.unique(ipix_all[from_target])
    if eval_mask is not None:
        ipix_ev = ipix_ev[eval_mask(ipix_ev)]
    n = int(ipix_ev.size)
    out = {k: np.full(n, np.nan) for k in ("ap_e", "sn_e", "ap_b", "sn_b")}
    out["counts"] = np.zeros(n, dtype=np.int32)
    out["sn_rand"] = np.full((n, n_random), np.nan)

    g1r = g2r = None
    if n_random:
        rng = np.random.default_rng(seed)
        ang = rng.uniform(0.0, np.pi, size=(ra_deg.size, n_random))
        c2, s2 = np.cos(2 * ang), np.sin(2 * ang)
        g1r = g1[:, None] * c2 - g2[:, None] * s2
        g2r = g1[:, None] * s2 + g2[:, None] * c2

    it = tqdm(enumerate(ipix_ev), total=n, desc="HEALPix cells") if progress else enumerate(ipix_ev)
    for k, ip in it:
        # Cell center, then all galaxies within the search radius of it.
        ra_c, dec_c = hp.pix2ang(nside, int(ip), nest=False, lonlat=True)
        # Unit-sphere KD-tree query: no RA = 0 seam, and a superset of the exact angular cut.
        idx = tree.query_ball_point(radec_to_xyz(ra_c, dec_c)[0], r=chord_radius(theta_max_deg))
        if not idx:
            continue
        idx = np.asarray(idx, dtype=np.int64)
        res = aperture_mass_and_sn_at_center(
            float(ra_c), float(dec_c), ra_deg[idx], dec_deg[idx], g1[idx], g2[idx],
            theta_max_arcmin=theta_max_arcmin, rs_deg=rs_deg, w_all=w_gal[idx], xc=xc,
            g1_rand=None if g1r is None else g1r[idx], g2_rand=None if g2r is None else g2r[idx],
        )
        out["ap_e"][k], out["sn_e"][k], out["ap_b"][k], out["sn_b"][k] = res[:4]
        if res[4] is not None:
            out["sn_rand"][k] = res[4]
        _d = ang_sep_deg(ra_deg[idx], dec_deg[idx], (ra_c, dec_c))
        out["counts"][k] = int(np.sum(_d <= theta_max_deg))
    out["ipix"] = ipix_ev
    out["completeness"] = out["counts"] / (N_DENSITY * np.pi * theta_max_arcmin**2)
    return out


# KD-tree on unit vectors (no RA = 0 seam); the exact angular cut is redone inside the aperture function.
tree = cKDTree(radec_to_xyz(ra_deg, dec_deg))

theta_max_arcmin = CONFIG["THETA_MAX_ARCMIN"]
rs_deg = float(CONFIG["RS_DEG"])
nside = int(CONFIG["NSIDE"])

amap = run_aperture_map(nside, rs_deg, theta_max_arcmin, n_random=CONFIG["N_RANDOM"], seed=CONFIG["RANDOM_SEED"])
ipix_eval = amap["ipix"]
ap_mass_map, sn_ratio_map = amap["ap_e"], amap["sn_e"]
ap_bmode_map, sn_bmode_map = amap["ap_b"], amap["sn_b"]
counts_map, completeness_map, sn_rand_map = amap["counts"], amap["completeness"], amap["sn_rand"]
n_cell = int(ipix_eval.size)

_good = completeness_map >= CONFIG["PEAK_COMPLETENESS_MIN"]
print("N HEALPix cells evaluated:", n_cell, f"({int(_good.sum())} with completeness >= {CONFIG['PEAK_COMPLETENESS_MIN']})")
print("Finite E-mode (aperture mass) cells:", int(np.isfinite(ap_mass_map).sum()))
print("Finite B-mode cells:", int(np.isfinite(ap_bmode_map).sum()))
print(f"std of S/N on complete cells:  E = {np.nanstd(sn_ratio_map[_good]):.2f}   B = {np.nanstd(sn_bmode_map[_good]):.2f}   "
      f"random rotations = {np.nanstd(sn_rand_map[_good]):.2f}")

### 4.4 Save the map

In [ ]:
# Sky coordinates of the evaluated cell centers.
ra_pix, dec_pix = hp.pix2ang(nside, ipix_eval, nest=False, lonlat=True)
ra_pix = np.asarray(ra_pix, dtype=np.float64)
dec_pix = np.asarray(dec_pix, dtype=np.float64)

In [ ]:
aper = Table(
    data=[ra_pix, dec_pix, ap_mass_map, ap_bmode_map, sn_ratio_map, sn_bmode_map, completeness_map],
    names=["RA", "DEC", "E_ap", "B_ap", "E_SN", "B_SN", "COMPLETENESS"],
)
aper.meta.update({"RS_ARCMIN": CONFIG["RS_ARCMIN"], "FILTER_XC": CONFIG["FILTER_XC"], "NSIDE": nside, "SHEAR_FRAME": CONFIG["SHEAR_FRAME"], "GX_SIGN": "standard", "DEREDDEN": CONFIG["DEREDDEN"]})
aper.write(OUTPUT_DIR / f"{CONFIG['TRACT']}_shapeHSM_massmap.fits", format="fits", overwrite=True)

## 5.0 Peaks

A peak is a cell whose E-mode S/N is the highest of all evaluated cells within `PEAK_EXCL_RS_MULT · Rs`, whose aperture
is at least `PEAK_COMPLETENESS_MIN` complete, and whose S/N is at least `PEAK_SN_MIN`. The threshold is chosen to sit
above the highest noise peak found in the random-rotation maps. Those maps include shape noise only. Cosmic shear adds
roughly 20% to the map scatter at Rs = 15′.

In [ ]:
sn_min = CONFIG.get("PEAK_SN_MIN")
excl_arcmin = CONFIG["PEAK_EXCL_RS_MULT"] * CONFIG["RS_ARCMIN"]
peak_kw = dict(nside=nside, excl_radius_arcmin=excl_arcmin, completeness_min=CONFIG["PEAK_COMPLETENESS_MIN"])

peak_ipix = find_peaks_shear_healpix(ipix_eval, sn_ratio_map, completeness_map, sn_min=sn_min, **peak_kw)
print("N peaks (with current cuts):", int(peak_ipix.size))

# Noise peaks in each random-rotation realization (no threshold), for the false-detection rate.
area_good = float(np.sum(completeness_map >= CONFIG["PEAK_COMPLETENESS_MIN"])) * hp.nside2pixarea(nside, degrees=True)
_idx_of = {int(p): k for k, p in enumerate(ipix_eval)}
rand_max, rand_n_above = [], []
for j in range(sn_rand_map.shape[1]):
    pk_r = find_peaks_shear_healpix(ipix_eval, sn_rand_map[:, j], completeness_map, sn_min=None, **peak_kw)
    vals = sn_rand_map[[_idx_of[int(p)] for p in pk_r], j]
    rand_max.append(np.nanmax(vals))
    rand_n_above.append(int(np.sum(vals >= (sn_min or 0))))
rand_max = np.asarray(rand_max)
print(f"Noise-only maps ({sn_rand_map.shape[1]} realizations, {area_good:.2f} deg^2 complete):")
print(f"  highest noise peak per tract: mean {rand_max.mean():.2f}, max {rand_max.max():.2f}")
print(f"  false peaks with S/N >= {sn_min}: {np.mean(rand_n_above):.2f} per tract = {np.mean(rand_n_above) / area_good:.2f} per deg^2")

# Peak catalog, highest S/N first.
_k_pk = np.array([_idx_of[int(p)] for p in peak_ipix], dtype=np.int64)
_ra_pk, _dec_pk = hp.pix2ang(nside, peak_ipix, nest=False, lonlat=True)
peaks_tbl = Table(
    data=[peak_ipix, _ra_pk, _dec_pk, sn_ratio_map[_k_pk], sn_bmode_map[_k_pk], ap_mass_map[_k_pk], completeness_map[_k_pk]],
    names=["IPIX", "RA", "DEC", "E_SN", "B_SN", "E_ap", "COMPLETENESS"],
)
peaks_tbl.meta.update({"TRACT": CONFIG["TRACT"], "RS_ARCMIN": CONFIG["RS_ARCMIN"], "NSIDE": nside, "SN_MIN": sn_min, "DEREDDEN": CONFIG["DEREDDEN"]})
peaks_tbl.write(OUTPUT_DIR / f"{CONFIG['TRACT']}_shapeHSM_peaks.fits", format="fits", overwrite=True)
peaks_tbl


## 6.0 Figures

### 6.1 Plot setup

In [ ]:
# Recompute the cell centers so the figures can be run without the save step.
ra_pix, dec_pix = hp.pix2ang(nside, ipix_eval, nest=False, lonlat=True)
ra_pix = np.asarray(ra_pix, dtype=np.float64)
dec_pix = np.asarray(dec_pix, dtype=np.float64)

# Reference RA for plotting; ra_plot is continuous across RA = 0 (tick labels are wrapped by ra_axis).
RA_PLOT_REF = circular_mean_ra(ra_pix)
ra_plot = ra_for_plot(ra_pix, RA_PLOT_REF)

# Only plot cells with at least one galaxy in the aperture.
msk = counts_map > 0
if msk.any():
    m_lo, m_hi = np.nanpercentile(ap_mass_map[msk], [5, 95])
    sn_lo, sn_hi = np.nanpercentile(sn_ratio_map[msk], [5, 95])
    b_lo, b_hi = np.nanpercentile(ap_bmode_map[msk], [5, 95])
    snb_lo, snb_hi = np.nanpercentile(sn_bmode_map[msk], [5, 95])
else:
    m_lo = m_hi = sn_lo = sn_hi = b_lo = b_hi = snb_lo = snb_hi = None

# Marker size shrinks as the number of plotted cells grows.
pt_size = max(1.0, min(8.0, 500_000.0 / max(int(ipix_eval.size), 1)))

# gnomview needs a full-sky array, so skip it if that would be too large.
MAX_DENSE_NPIX = 4_000_000
_npix_hp = hp.nside2npix(nside)
do_gnomonic = _npix_hp <= MAX_DENSE_NPIX
if do_gnomonic:
    # Center the projection on the data.
    rot_ra = RA_PLOT_REF
    rot_dec = float(np.median(dec_pix))

    # Scatter the E-mode S/N onto the full sky, marking empty pixels as UNSEEN.
    sn_dense = np.full(_npix_hp, np.nan, dtype=np.float64)
    sn_dense[ipix_eval] = sn_ratio_map
    hp_sm = np.copy(sn_dense)
    hp_sm[~np.isfinite(hp_sm)] = hp.UNSEEN

    # Same for the B-mode S/N.
    snb_dense = np.full(_npix_hp, np.nan, dtype=np.float64)
    snb_dense[ipix_eval] = sn_bmode_map
    hp_smb = np.copy(snb_dense)
    hp_smb[~np.isfinite(hp_smb)] = hp.UNSEEN
else:
    hp_sm = hp_smb = None

### 6.2 E-mode maps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)

# Left: aperture E-mode signal.
sc0 = axes[0].scatter(
    ra_plot[msk],
    dec_pix[msk],
    c=ap_mass_map[msk],
    s=pt_size,
    cmap="RdBu_r",
    vmin=m_lo,
    vmax=m_hi,
    linewidths=0,
    alpha=0.85,
)
ra_axis(axes[0])
axes[0].set_ylabel("Dec (deg)")
axes[0].set_title(r"Aperture E-mode (gt×Q)")
fig.colorbar(sc0, ax=axes[0], fraction=0.046, pad=0.02)

# Middle: the same quantity divided by its shape-noise error.
sc1 = axes[1].scatter(
    ra_plot[msk],
    dec_pix[msk],
    c=sn_ratio_map[msk],
    s=pt_size,
    cmap="RdBu_r",
    vmin=-3.0,
    vmax=3.0,
    linewidths=0,
    alpha=0.85,
)
ra_axis(axes[1])
axes[1].set_ylabel("Dec (deg)")
axes[1].set_title(r"E-mode S/N")
fig.colorbar(sc1, ax=axes[1], fraction=0.046, pad=0.02)

# Right: source density per aperture, offset before the log to keep empty cells finite.
logc = np.log10(counts_map.astype(np.float64) + 0.1)
sc2 = axes[2].scatter(
    ra_plot[msk],
    dec_pix[msk],
    c=logc[msk],
    s=pt_size,
    cmap="magma",
    linewidths=0,
    alpha=0.85,
)
ra_axis(axes[2])
axes[2].set_ylabel("Dec (deg)")
axes[2].set_title(r"$\log_{10}$(counts in search disc) + 0.1")
fig.colorbar(sc2, ax=axes[2], fraction=0.046, pad=0.02)


fig.suptitle(
    f"Tract {CONFIG['TRACT']}, NSIDE={nside}, Rs={CONFIG['RS_ARCMIN']:.2f}', search={theta_max_arcmin:.1f}'",
    y=1.02,
)
save_plot(fig, f"{CONFIG['TRACT']}_E-mode_map")

### 6.3 B-mode maps

Structure here at the level of the E-mode maps would indicate systematics rather than lensing.

In [ ]:
fig_eb, axes_eb = plt.subplots(1, 2, figsize=(10.5, 4.2), constrained_layout=True)

# Left: aperture B-mode signal.
scb0 = axes_eb[0].scatter(
    ra_plot[msk],
    dec_pix[msk],
    c=ap_bmode_map[msk],
    s=pt_size,
    cmap="RdBu_r",
    vmin=b_lo,
    vmax=b_hi,
    linewidths=0,
    alpha=0.85,
)
ra_axis(axes_eb[0])
axes_eb[0].set_ylabel("Dec (deg)")
axes_eb[0].set_title(r"Aperture B-mode (g××Q)")
fig_eb.colorbar(scb0, ax=axes_eb[0], fraction=0.046, pad=0.02)

# Right: B-mode S/N.
scb1 = axes_eb[1].scatter(
    ra_plot[msk],
    dec_pix[msk],
    c=sn_bmode_map[msk],
    s=pt_size,
    cmap="RdBu_r",
    vmin=-3.0,
    vmax=3.0,
    linewidths=0,
    alpha=0.85,
)
ra_axis(axes_eb[1])
axes_eb[1].set_ylabel("Dec (deg)")
axes_eb[1].set_title(r"B-mode S/N")
fig_eb.colorbar(scb1, ax=axes_eb[1], fraction=0.046, pad=0.02)


fig_eb.suptitle(
    f"Tract {CONFIG['TRACT']}, same filter/weights as E-mode row above",
    y=1.02,
)
save_plot(fig_eb, f"{CONFIG['TRACT']}_B-mode_map")

In [ ]:
# E-mode against B-mode aperture values per occupied cell.
plt.hexbin(ap_mass_map[msk], ap_bmode_map[msk], mincnt=1)
plt.colorbar()
plt.gca().set_aspect(1)
save_plot(plt.gcf(), f"{CONFIG['TRACT']}_EB_hexbin")

In [ ]:
# Histogram of E-mode and B-mode aperture values per occupied cell.
# B should look like pure noise; an E-mode tail beyond it is the lensing signal.
e_vals = ap_mass_map[msk]
b_vals = ap_bmode_map[msk]
e_vals = e_vals[np.isfinite(e_vals)]
b_vals = b_vals[np.isfinite(b_vals)]
eb_bins = np.histogram_bin_edges(np.concatenate([e_vals, b_vals]), bins=60)

fig_hist, ax_hist = plt.subplots(figsize=(6, 4.2), constrained_layout=True)
ax_hist.hist(e_vals, bins=eb_bins, histtype="step", linewidth=1.5, label="E-mode")
ax_hist.hist(b_vals, bins=eb_bins, histtype="step", linewidth=1.5, label="B-mode")
ax_hist.set_xlabel("Aperture mass")
ax_hist.set_ylabel("Number of cells")
ax_hist.legend()
save_plot(fig_hist, f"{CONFIG['TRACT']}_EB_hist")

### 6.4 Peaks

In [ ]:
fig2, axp = plt.subplots(figsize=(7, 5.5))

# E-mode S/N on a fixed -4 to 4 scale so peak significance is directly readable.
sc = axp.scatter(
    ra_plot[msk],
    dec_pix[msk],
    c=sn_ratio_map[msk],
    s=pt_size,
    cmap="RdBu_r",
    vmin=-4,
    vmax=4,
    linewidths=0,
    alpha=0.85,
)

# Circle the detected peaks.
if peak_ipix.size:
    ra_peak, dec_peak = hp.pix2ang(nside, peak_ipix, nest=False, lonlat=True)
    axp.scatter(
        ra_for_plot(ra_peak, RA_PLOT_REF),
        dec_peak,
        s=120,
        facecolors="none",
        edgecolors="k",
        linewidths=1.2,
        label="peaks",
    )

ra_axis(axp)
axp.set_ylabel("Dec (deg)")

axp.grid()
axp.set_title(r"E-mode $\mathcal{S}/\mathcal{N}$ with peak positions")
fig2.colorbar(sc, ax=axp, fraction=0.046, pad=0.02)
if peak_ipix.size:
    axp.legend(loc="lower left", framealpha=1)
fig2.tight_layout()
save_plot(fig2, f"{CONFIG['TRACT']}_E-mode_sn_ratio")

### 6.5 Known clusters and groups

Clusters and groups from SIMBAD in the tract, with masses from SZ and X-ray catalogs (PSZ2, ACT DR5, SPT-SZ, eRASS1, MCXC) on VizieR, overlaid on the E-mode S/N map. Mass definitions differ between catalogs, so each label names its definition and catalog. The table is kept as `known_clusters`, with the map S/N at each object and its distance to the nearest detected peak. Needs network access.

In [ ]:
# Known galaxy clusters and groups on the E-mode S/N map.
# Positions, types and redshifts come from SIMBAD. SIMBAD stores no cluster masses, so masses come from
# SZ / X-ray catalogs on VizieR, matched to the SIMBAD objects by position. Needs network access.
import re

import astropy.units as u
import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
from astropy.coordinates import SkyCoord
from astropy.table import Table
from astroquery.simbad import Simbad
from astroquery.vizier import Vizier

CL_CFG = {
    # SIMBAD object types: cluster, group, compact group, proto-cluster, and their candidates.
    "OTYPES": ["ClG", "GrG", "CGG", "PCG"],
    "CANDIDATE_OTYPES": ["C?G", "Gr?", "PCG?"],
    "INCLUDE_CANDIDATES": False,  # candidates are numerous (photometric catalogs); markers only, never labeled
    "Z_MAX": None,                # e.g. 1.2 to drop objects above that redshift (objects without z are kept)
    "MIN_NBREF": 0,               # minimum number of SIMBAD references
    "MATCH_ARCMIN": 2.0,          # SIMBAD <-> mass-catalog match radius
    "N_LABEL": 12,                # annotate at most this many objects (those with a mass first)
    # Mass catalogs on VizieR, in order of preference when several match the same object.
    "MASS_CATALOGS": {
        "PSZ2": "J/A+A/594/A27/psz2",
        "ACT DR5": "J/ApJS/253/3",
        "SPT-SZ": "J/ApJS/216/27",
        "eRASS1": "J/A+A/685/A106",
        "MCXC": "J/A+A/534/A109",
    },
}

# Column-name patterns (case-insensitive, full match, first hit wins); the chosen columns are printed.
_MASS_PATTERNS = [r"M500c?Cal", r"M500c?", r"MSZ", r"M500\w*", r"M200[mc]?", r"Mass"]
_Z_PATTERNS = [r"z", r"zbest", r"best_?z", r"z_?spec", r"z_?cl", r"z_?ph(ot)?", r"redshift"]


def _num(col):
    """Column as float with masked/invalid entries as NaN."""
    return np.ma.filled(np.ma.asarray(col).astype(float), np.nan)


def _pick(tbl, patterns):
    for pat in patterns:
        for name in tbl.colnames:
            if re.fullmatch(pat, name, flags=re.I) and tbl[name].dtype.kind in "fiu":
                return name
    return None


def _mass_msun(col):
    """Mass column converted to Msun from its VizieR unit; assumes 1e14 Msun if the unit is missing."""
    v = _num(col)
    if col.unit is not None:
        try:
            return (v * col.unit).to_value(u.solMass), str(col.unit)
        except Exception:
            pass
    return v * 1e14, "no unit, assumed 1e14 Msun"


def _wrap_ra(ra, ref):
    return ref + ((np.asarray(ra, dtype=np.float64) - ref + 180.0) % 360.0 - 180.0)


# Region covered by the evaluated map cells.
_ra_cell, _dec_cell = hp.pix2ang(nside, ipix_eval, nest=False, lonlat=True)
_ra_rad = np.deg2rad(_ra_cell)
_ra0 = float(np.rad2deg(np.arctan2(np.sin(_ra_rad).mean(), np.cos(_ra_rad).mean())) % 360.0)
_dec0 = float(np.mean(_dec_cell))
_cells = SkyCoord(_ra_cell * u.deg, _dec_cell * u.deg)
_center = SkyCoord(_ra0 * u.deg, _dec0 * u.deg)
_radius_deg = float(_cells.separation(_center).max().deg) + 0.1
print(f"Search circle: RA={_ra0:.3f} Dec={_dec0:.3f} r={_radius_deg:.2f} deg")

# 1. SIMBAD: clusters and groups in the region.
_otypes = CL_CFG["OTYPES"] + (CL_CFG["CANDIDATE_OTYPES"] if CL_CFG["INCLUDE_CANDIDATES"] else [])
_adql = f"""
SELECT main_id, ra, dec, otype, rvz_redshift, nbref
FROM basic
WHERE otype IN ({", ".join(repr(o) for o in _otypes)})
  AND CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', {_ra0:.6f}, {_dec0:.6f}, {_radius_deg:.4f})) = 1
"""
_sim = Simbad.query_tap(_adql)
print(f"SIMBAD: {len(_sim)} objects of type {', '.join(_otypes)}")

objs = Table(
    {
        "name": np.array([str(s) for s in _sim["main_id"]], dtype="U64"),
        "otype": np.array([str(s) for s in _sim["otype"]], dtype="U16"),
        "ra": _num(_sim["ra"]),
        "dec": _num(_sim["dec"]),
        "z": _num(_sim["rvz_redshift"]),
        "nbref": np.asarray(_num(_sim["nbref"])),
        "mass": np.full(len(_sim), np.nan),
        "mass_def": np.full(len(_sim), "", dtype="U24"),
        "mass_cat": np.full(len(_sim), "", dtype="U32"),
    }
)

# 2. VizieR: masses (and redshifts) from SZ / X-ray catalogs in the same circle.
_viz = Vizier(columns=["*", "_RAJ2000", "_DEJ2000"], row_limit=-1)
_mass_rows = []
for _label, _cat in CL_CFG["MASS_CATALOGS"].items():
    try:
        _res = _viz.query_region(_center, radius=_radius_deg * u.deg, catalog=_cat)
    except Exception as exc:
        print(f"{_label} ({_cat}): query failed: {exc}")
        continue
    if len(_res) == 0:
        print(f"{_label} ({_cat}): no rows in this region")
        continue
    for _t in _res:
        _mcol, _zcol = _pick(_t, _MASS_PATTERNS), _pick(_t, _Z_PATTERNS)
        if _mcol is None:
            print(f"{_label} [{_t.meta.get('name', '')}]: {len(_t)} rows but no mass column found")
            continue
        _m, _unit = _mass_msun(_t[_mcol])
        _z = _num(_t[_zcol]) if _zcol else np.full(len(_t), np.nan)
        # Object name: first text column that is not a VizieR computed column (_RAJ2000, ...).
        _name = next((c for c in _t.colnames if not c.startswith("_") and _t[c].dtype.kind in "SUO"), None)
        print(f"{_label} [{_t.meta.get('name', '')}]: {len(_t)} rows, mass={_mcol} [{_unit}], z={_zcol}")
        for k in range(len(_t)):
            _id = str(_t[_name][k]).strip() if _name else f"#{k}"
            _mass_rows.append((_label, f"{_label} {_id}", float(_t["_RAJ2000"][k]),
                               float(_t["_DEJ2000"][k]), _z[k], _m[k], _mcol))

# 3. Match: each SIMBAD object takes the mass of the preferred catalog entry within MATCH_ARCMIN.
#    Catalog entries with no SIMBAD counterpart are added as objects of their own.
_order = {lab: i for i, lab in enumerate(CL_CFG["MASS_CATALOGS"])}
_mass_rows.sort(key=lambda r: _order[r[0]])
_extra = []
_sc_obj = SkyCoord(objs["ra"] * u.deg, objs["dec"] * u.deg) if len(objs) else None
for lab, cname, ra_, dec_, z_, m_, mdef in _mass_rows:
    if not np.isfinite(m_):
        continue
    j = None
    if _sc_obj is not None:
        _s = _sc_obj.separation(SkyCoord(ra_ * u.deg, dec_ * u.deg)).arcmin
        if _s.min() <= CL_CFG["MATCH_ARCMIN"]:
            j = int(np.argmin(_s))
    if j is None:
        _extra.append((cname, lab, ra_, dec_, z_, 0.0, m_, mdef, lab))
        continue
    if not np.isfinite(objs["mass"][j]):  # keep the preferred catalog's mass
        objs["mass"][j], objs["mass_def"][j], objs["mass_cat"][j] = m_, mdef, lab
        if not np.isfinite(objs["z"][j]):
            objs["z"][j] = z_
for row in _extra:
    objs.add_row(row)

# Keep objects on the evaluated map and apply the cuts.
if len(objs):
    _sc = SkyCoord(objs["ra"] * u.deg, objs["dec"] * u.deg)
    _, _d2, _ = _sc.match_to_catalog_sky(_cells)
    _keep = _d2.deg <= 2.0 * np.sqrt(hp.nside2pixarea(nside, degrees=True))
    _keep &= objs["nbref"] >= CL_CFG["MIN_NBREF"]
    if CL_CFG["Z_MAX"] is not None:
        _keep &= ~(objs["z"] > CL_CFG["Z_MAX"])
    objs = objs[_keep]

# E-mode S/N of the map cell at each object, and distance to the nearest detected peak.
_k_of = {int(p): k for k, p in enumerate(ipix_eval)}
_ip_obj = hp.ang2pix(nside, np.asarray(objs["ra"]), np.asarray(objs["dec"]), nest=False, lonlat=True)
objs["E_SN"] = [sn_ratio_map[_k_of[int(p)]] if int(p) in _k_of else np.nan for p in _ip_obj]
objs["peak_sep_arcmin"] = np.nan
if "peak_ipix" in globals() and len(peak_ipix) and len(objs):
    _ra_pk, _dec_pk = hp.pix2ang(nside, peak_ipix, nest=False, lonlat=True)
    _, _d2p, _ = SkyCoord(objs["ra"] * u.deg, objs["dec"] * u.deg).match_to_catalog_sky(
        SkyCoord(_ra_pk * u.deg, _dec_pk * u.deg))
    objs["peak_sep_arcmin"] = _d2p.arcmin

# Most interesting first: objects with a mass (heaviest first), then by number of references.
objs = objs[np.lexsort((-objs["nbref"], -np.nan_to_num(objs["mass"], nan=-1.0)))]
known_clusters = objs  # kept for later cells
_show = objs["name", "otype", "ra", "dec", "z", "mass", "mass_def", "mass_cat", "E_SN", "peak_sep_arcmin"]
for c in ("ra", "dec"):
    _show[c].format = ".4f"
for c in ("z", "E_SN", "peak_sep_arcmin"):
    _show[c].format = ".2f"
_show["mass"].format = ".2e"
_show.pprint(max_lines=40, max_width=200)

# 4. Plot: E-mode S/N map with the objects overlaid.
_ref = float(globals().get("RA_PLOT_REF", _ra0))
_msk = counts_map > 0
_pt = max(1.0, min(8.0, 500_000.0 / max(int(ipix_eval.size), 1)))
fig_cl, ax_cl = plt.subplots(figsize=(8.5, 7))
_scm = ax_cl.scatter(_wrap_ra(_ra_cell, _ref)[_msk], _dec_cell[_msk], c=sn_ratio_map[_msk], s=_pt,
                     cmap="RdBu_r", vmin=-4, vmax=4, linewidths=0, alpha=0.85)
if "peak_ipix" in globals() and len(peak_ipix):
    _ra_pk, _dec_pk = hp.pix2ang(nside, peak_ipix, nest=False, lonlat=True)
    ax_cl.scatter(_wrap_ra(_ra_pk, _ref), _dec_pk, s=140, facecolors="none", edgecolors="k",
                  linewidths=1.2, label="S/N peaks")

_style = {  # otype -> (marker, color, legend label)
    "ClG": ("s", "darkviolet", "cluster (SIMBAD)"),
    "GrG": ("^", "darkgreen", "group (SIMBAD)"),
    "CGG": ("^", "darkgreen", "group (SIMBAD)"),
    "PCG": ("D", "darkorange", "proto-cluster (SIMBAD)"),
}
_cand = ("0.45", "candidate (SIMBAD)")
_used = set()
for k in range(len(objs)):
    ot = objs["otype"][k]
    if ot in _style:
        mk, col, lab = _style[ot]
    elif ot in CL_CFG["MASS_CATALOGS"]:
        mk, col, lab = "P", "goldenrod", "SZ / X-ray catalog only"
    else:
        mk, (col, lab) = "x", _cand
    _colors = dict(c=col) if mk == "x" else dict(facecolors="none", edgecolors=col)
    ax_cl.scatter(_wrap_ra(objs["ra"][k], _ref), objs["dec"][k], marker=mk, s=70, linewidths=1.6,
                  label=None if lab in _used else lab, zorder=4, **_colors)
    _used.add(lab)

# Labels: name, redshift and mass for the top N objects (candidates are never labeled).
_n = 0
for k in range(len(objs)):
    if _n >= CL_CFG["N_LABEL"] or objs["otype"][k] in CL_CFG["CANDIDATE_OTYPES"]:
        continue
    if not (np.isfinite(objs["z"][k]) or np.isfinite(objs["mass"][k])):
        continue
    txt = objs["name"][k]
    if np.isfinite(objs["z"][k]):
        txt += f"\nz = {objs['z'][k]:.3f}"
    if np.isfinite(objs["mass"][k]):
        txt += f"\n{objs['mass_def'][k]} = {objs['mass'][k] / 1e14:.2f}e14 M$_\\odot$ ({objs['mass_cat'][k]})"
    ax_cl.annotate(txt, (_wrap_ra(objs["ra"][k], _ref), objs["dec"][k]), xytext=(6, 6),
                   textcoords="offset points", fontsize=7, zorder=5,
                   bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="0.6", alpha=0.8))
    _n += 1

if "ra_axis" in globals():
    ra_axis(ax_cl)
else:
    ax_cl.set_xlabel("RA (deg)")
    ax_cl.invert_xaxis()
ax_cl.set_ylabel("Dec (deg)")
ax_cl.grid(alpha=0.3)
_tract = globals().get("CONFIG", {}).get("TRACT", "")
ax_cl.set_title(rf"Tract {_tract}: E-mode $\mathcal{{S}}/\mathcal{{N}}$ with known clusters and groups")
fig_cl.colorbar(_scm, ax=ax_cl, fraction=0.046, pad=0.02, label="E-mode S/N")
ax_cl.legend(loc="lower left", fontsize=8, framealpha=1)
fig_cl.tight_layout()
if "save_plot" in globals():
    save_plot(fig_cl, f"{_tract}_E-mode_sn_known_clusters")